# Lab 02: k-NN Distance Computation
We implement the squared Euclidean distance between every training and test sample in three ways (two loops, one loop, no loops), verify they agree, and compare their runtime.

In [ ]:
import time
import torch
import matplotlib.pyplot as plt
import dlcv2026

plt.rcParams["figure.figsize"] = (10.0, 8.0)
plt.rcParams["font.size"] = 16

from knn import (hello, compute_distances_two_loops,
                 compute_distances_one_loop, compute_distances_no_loops)
hello()

## 1. Hand-checkable test
Training points (0,0), (3,4); test points (0,4), (3,0). The squared distances are easy to compute by hand.

In [ ]:
x_train_small = torch.tensor([[0.0, 0.0], [3.0, 4.0]], dtype=torch.float64)
x_test_small = torch.tensor([[0.0, 4.0], [3.0, 0.0]], dtype=torch.float64)
expected = torch.tensor([[16.0, 9.0], [9.0, 16.0]], dtype=torch.float64)

for fn in [compute_distances_two_loops, compute_distances_one_loop, compute_distances_no_loops]:
    actual = fn(x_train_small, x_test_small)
    assert torch.allclose(actual, expected), f"{fn.__name__}: expected {expected}, got {actual}"
    print(fn.__name__, "passed")

## 2. Load a CIFAR-10 subset

In [ ]:
torch.manual_seed(0)
num_train = 500
num_test = 100
x_train, y_train, x_test, y_test = dlcv2026.data.cifar10(num_train, num_test)
print("Train data shape:", x_train.shape)
print("Test data shape:", x_test.shape)

## 3. Two loops

In [ ]:
x_train_before, x_test_before = x_train.clone(), x_test.clone()
dists_two = compute_distances_two_loops(x_train, x_test)
print("shape:", dists_two.shape, "dtype:", dists_two.dtype)
assert dists_two.shape == (num_train, num_test)
assert dists_two.dtype == x_train.dtype
assert torch.equal(x_train, x_train_before) and torch.equal(x_test, x_test_before), "inputs were modified"
print("dists[0, :5] =", dists_two[0, :5])

## 4. One loop (must match two loops)

In [ ]:
dists_one = compute_distances_one_loop(x_train, x_test)
diff = (dists_one - dists_two).abs().max().item()
print("max abs difference vs two loops:", diff)
assert torch.allclose(dists_one, dists_two, atol=1e-4), "one loop does not match two loops"
print("one loop matches two loops")

## 5. No loops (must match two loops)

In [ ]:
dists_none = compute_distances_no_loops(x_train, x_test)
diff = (dists_none - dists_two).abs().max().item()
print("max abs difference vs two loops:", diff)
assert torch.allclose(dists_none, dists_two, atol=1e-3), "no loops does not match two loops"
assert (dists_none >= 0).all()
print("no loops matches two loops")

## 6. Other input shapes
The functions must also work for plain vectors, not only images.

In [ ]:
torch.manual_seed(1)
a = torch.randn(7, 5, dtype=torch.float64)
b = torch.randn(4, 5, dtype=torch.float64)
d2, d1, d0 = (compute_distances_two_loops(a, b),
              compute_distances_one_loop(a, b),
              compute_distances_no_loops(a, b))
assert torch.allclose(d2, d1) and torch.allclose(d2, d0)
print("vector inputs OK, shape", d0.shape)

## 7. Runtime comparison

In [ ]:
def timeit(f, *args, n_runs=3):
    best = float("inf")
    for _ in range(n_runs):
        t0 = time.perf_counter()
        f(*args)
        best = min(best, time.perf_counter() - t0)
    return best

t_two = timeit(compute_distances_two_loops, x_train, x_test)
t_one = timeit(compute_distances_one_loop, x_train, x_test)
t_none = timeit(compute_distances_no_loops, x_train, x_test)
print(f"two loops : {t_two:.4f} s")
print(f"one loop  : {t_one:.4f} s  ({t_two / t_one:.1f}x faster than two loops)")
print(f"no loops  : {t_none:.4f} s  ({t_two / t_none:.1f}x faster than two loops)")

plt.bar(["two loops", "one loop", "no loops"], [t_two, t_one, t_none])
plt.yscale("log")
plt.ylabel("seconds (log scale)")
plt.title("Distance computation runtime")
plt.show()

## Wrap-up
**Why are the vectorized versions faster?**
A Python loop runs one tiny operation per iteration, so most of the time goes into interpreter overhead. With one loop, each iteration processes all test samples at once inside PyTorch's compiled code. With no loops, the whole computation becomes a single matrix multiplication plus broadcasts, which runs in optimized BLAS code and can use multiple cores or a GPU.

**How does the no-loop version work?**
Using ||a - b||^2 = ||a||^2 + ||b||^2 - 2 a.b, the cross term for all pairs is one matrix product `x_train @ x_test.T`, and the squared norms are added with broadcasting. This avoids building the huge (num_train, num_test, D) difference tensor.

**Numbers from my run** (fill in): two loops ____ s, one loop ____ s, no loops ____ s.